## Reading 35-1 - The Power of Ten, Rules 4 through 7

By the end of this reading, you will learn:
<ol>
    <li>Rule 4: why functions should fit on one page</li>
    <li>Rule 5: why <code>assert()</code> is not good enough for flight software, and what to use instead</li>
    <li>Rule 6: how the <code>-Wshadow</code> flag you have used all semester enforces a Power of Ten rule</li>
    <li>Rule 7: how to make the compiler refuse to build code that ignores an error</li>
</ol>

### Getting the Reading Code

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your Reading Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir reading35
    cd reading35

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/readings/lec35/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    assert_ndebug.c  cmd_queue.c  limit.txt  macros.c  Makefile  nodiscard.c  recovery_assert.c  shadow.c

### Where We Left Off

In Lecture 34, we applied the first three rules to a spacecraft command queue: no recursion, a fixed bound on every loop, and no <code>malloc</code> after startup. Those three rules make sure the program's <i>resources</i> (stack, time, and memory) are bounded. Rules 4 through 7 are about something different: making sure that when something goes wrong, <b>the program notices</b>.

### Rule 4 - Functions Fit on One Page

Holzmann's rule: no function should be longer than what can be printed on a single sheet of paper, about 60 lines of code with one statement per line.

The reasoning is human, not technical. A function you can see all at once is a function you can review, understand, and test as a unit. A 400-line function has too many paths through it for anyone to check by hand, and usually means the code was never broken into clear steps.

You have been following this rule without being told. Every assignment this semester has split the work into small functions in separate <code>.c</code> files, like <code>labyrinth.c</code> and <code>maze_generator.c</code> in Homework 8, or <code>matmul.c</code> and <code>epsilon.c</code> in Homework 9.

### Rule 5 - Assertions That Recover

An <b>assertion</b> is a check for something that should <i>never</i> happen if the code is correct. C provides <code>assert()</code> in <code>&lt;assert.h&gt;</code>: if the condition is false, the program prints a message and stops.

Holzmann's rule asks for at least two assertions per function, on average. But he does <i>not</i> mean the standard <code>assert()</code>, for two reasons.

<b>Reason 1: <code>assert()</code> stops the program.</b> A spacecraft cannot stop. Here is <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec35/assert_ndebug.c">assert_ndebug.c</a>, which loads a thruster temperature limit from a file:

    double max_temp = 0.0;

    /* Load the limit, and stop the program if loading fails */
    assert(load_limit(argv[1], &max_temp) == 0);

    $ ./assert_ndebug missing.txt
    assert_ndebug: assert_ndebug.c:28: main: Assertion `load_limit(argv[1], &max_temp) == 0' failed.
    Aborted

That is reasonable on a laptop. On a flight computer, aborting means losing control of the vehicle.

<b>Reason 2: <code>assert()</code> can disappear.</b> If the macro <code>NDEBUG</code> is defined, every <code>assert()</code> in the program is removed by the preprocessor, <i>including the code inside it</i>. Many projects define <code>NDEBUG</code> for their "release" build. Watch what happens with a configuration file that is perfectly fine:

    $ make asserts
    ./assert_ndebug limit.txt
    Limit: 450.0 C   Reading: 285.0 C   Nominal
    ./assert_ndebug_off limit.txt
    Limit: 0.0 C   Reading: 285.0 C   OVER LIMIT: shutting down thruster

The second build was compiled with <code>-DNDEBUG</code>. The call to <code>load_limit</code> was <i>inside</i> the assertion, so it was removed along with it. The file was never read, the limit stayed at 0.0, and a nominal reading shut down the thruster. The version you tested and the version you flew behaved differently, the same lesson as Reading 32-2.

<b>Never put code that must run inside <code>assert()</code>.</b> Holzmann's rule says this directly: every assertion must be <i>side-effect free</i>, meaning it only checks values and never changes them.

### Holzmann's Recovery Assertion - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec35/recovery_assert.c">recovery_assert.c</a>

Holzmann's paper defines an assertion that reports the problem and then lets the caller <i>recover</i>:

    #define c_assert(e) ((e) ? true : \
    	(fprintf(stderr, "%s:%d: assertion failed: %s\n", __FILE__, __LINE__, #e), false))

<ul>
    <li>If <code>e</code> is true, the whole expression is <code>true</code>.</li>
    <li>If <code>e</code> is false, it prints the file, line, and the condition itself (<code>#e</code> turns the expression into a string), and the whole expression is <code>false</code>.</li>
</ul>

The function that uses it checks its own parameters and returns an error code instead of stopping:

    int average_temp(const double readings[], int count, double *average)
    {
    	if (!c_assert(readings != NULL) || !c_assert(average != NULL)) {
    		return ERR_PARAM;
    	}
    	if (!c_assert(count > 0 && count <= MAX_READINGS)) {
    		return ERR_PARAM;
    	}
    	...

    $ ./recovery_assert
    Average of 4 readings: 282.75 C
    recovery_assert.c:20: assertion failed: count > 0 && count <= MAX_READINGS
    Bad reading count. Using last known good average: 282.75 C
    Still running.

The bad call was detected, reported with its exact location, and handled. The program kept running. That is the difference Holzmann wants: an assertion is not a crash, it is a <b>detection</b> followed by a <b>recovery action</b> that you designed in advance.

### Rule 6 - Declare Data at the Smallest Possible Scope - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec35/shadow.c">shadow.c</a>

If a variable is only used inside one loop, declare it inside that loop. If it is only used in one function, declare it in that function. The less code that can see a variable, the less code that can change it by accident.

One of the most common scope bugs is <i>shadowing</i>: declaring a local variable with the same name as a global one, so the function quietly uses the wrong variable:

    int count = 0;	/* number of commands executed, visible to the whole file */

    void execute_commands(int num_commands)
    {
    	int count = 0;	/* meant to be a loop counter, but it hides the global */
    	...

    $ make shadow
    shadow.c:8:13: error: declaration of 'count' shadows a global declaration [-Werror=shadow]

This is why <code>-Wshadow</code> has been in the course flags since Homework 1. You have been following Rule 6 with a tool's help all semester.

### Rule 7 - Check Every Return Value - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec35/nodiscard.c">nodiscard.c</a>

Rule 7 has two halves:
<ul>
    <li>The <b>caller</b> must check the return value of every non-<code>void</code> function</li>
    <li>Every <b>function</b> must check the validity of its parameters, as <code>average_temp</code> does above</li>
</ul>

Ignoring a return value is the most common way a failure turns silent. The C23 standard, which we use through <code>-std=c2x</code>, gives you a way to make the compiler enforce the first half. Mark a function <code>[[nodiscard]]</code>, and any caller that ignores its result gets a warning:

    [[nodiscard]] int read_sensor(int channel, double *value)
    {
    	if (channel < 0 || channel > 3) {
    		return ERR_SENSOR;	/* there are only 4 channels */
    	}
    	...
    }

    int main(void)
    {
    	double temp = 0.0;

    	/* Channel 7 does not exist, and we never check */
    	read_sensor(7, &temp);
    	...

    $ make nodiscard
    nodiscard.c:22:9: error: ignoring return value of 'read_sensor', declared with attribute 'nodiscard' [-Werror=unused-result]

Because of <code>-Werror</code>, the program does not build. The mistake is caught before the code ever runs.

What if you truly do not need a return value? Holzmann's rule allows you to say so explicitly by casting it to <code>void</code>, for example <code>(void)fprintf(...)</code>. You have seen this already: the starter code in this course uses <code>(void)argc;</code> to tell the compiler that an unused parameter is intentional. The point is not that every value must be used. The point is that ignoring one must be a <i>decision</i>, written in the code where a reviewer can see it.

### <font color = "red">Class Introduction Question #1 - Give two reasons that the standard <code>assert()</code> is not acceptable in flight software, and explain why <code>assert_ndebug_off</code> shut down the thruster on a nominal reading.</font>

### <font color = "red">Class Introduction Question #2 - How is Holzmann's <code>c_assert</code> different from <code>assert()</code>? What does the function that uses it do when an assertion fails?</font>

### <font color = "red">Class Introduction Question #3 - Which compiler flag in our course enforces Rule 6, and what C23 feature can enforce the first half of Rule 7?</font>

### The next reading for this lecture is <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec35/Reading%2035-2.ipynb">Reading 35-2 - The Power of Ten, Rules 8 through 10, and the Tools That Check Them</a>